In [ ]:
import openai
import os
from pinecone import Pinecone, ServerlessSpec

# Coloque sua chaves aqui (ou use .env)
openai.api_key = ""

# Iniciar cliente
pc = Pinecone(api_key="")

index_name = "meu-index"

if index_name not in pc.list_indexes().names():
    pc.create_index(
        name=index_name,
        dimension=1536, # !
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        )
    )

index = pc.Index(index_name)



In [ ]:

from openai import OpenAI
client = OpenAI(api_key="")
def gerar_embedding(texto: str) -> list:
    response = client.embeddings.create(
        input=texto,
        model="text-embedding-3-small" # !
    )
    return response.data[0].embedding

documentos = [
    {"id": "doc1", "texto": "Curso: Astronomia - Preço: R$ 500 - Quantidade de Matérias: 20"},
    {"id": "doc2", "texto": "Curso: Python - Preço: R$ 1.259(Á vista) Parcelado:R$ 1.536 - Quantidade de Matérias: 56"},
]

vetores = []
for doc in documentos:
    embedding = gerar_embedding(doc["texto"])
    vetores.append({
        "id": doc["id"],
        "values": embedding,
        "metadata": {"texto": doc["texto"]},
    })

index.upsert(vectors=vetores)
print("Vetores enviados!")


Vetores enviados!


In [18]:
pergunta = "Quanto Custa o Curso de Python?"
embedding_pergunta = gerar_embedding(pergunta)

resultado = index.query(vector=embedding_pergunta, top_k=2, include_metadata=True)

# Mostrar resultados
for match in resultado["matches"]:
    print(f"\n🧠 Score: {match['score']:.4f}")
    print(f"📄 Texto: {match['metadata']['texto']}")



🧠 Score: 0.7009
📄 Texto: Curso: Python - Preço: R$ 1.259(Á vista) Parcelado:R$ 1.536 - Quantidade de Matérias: 56

🧠 Score: 0.4596
📄 Texto: Curso: Astronomia - Preço: R$ 500 - Quantidade de Matérias: 20


In [ ]:

# Pergunta
pergunta = "Quanto custa o curso de Python?"
embedding_pergunta = gerar_embedding(pergunta)

# Consulta no Pinecone
resultado = index.query(vector=embedding_pergunta, top_k=2, include_metadata=True)

print("Resultado: ", resultado) # Mostrar Retornos

# Montar contexto com os textos retornados da consulta
contexto = "\n\n".join([match["metadata"]["texto"] for match in resultado["matches"]])

# Prompt
prompt = f"""
Você é um assistente que responde perguntas com base em informações disponíveis.

Pergunta do usuário: "{pergunta}"

Baseado nas informações abaixo, elabore uma resposta clara e bem escrita:

{contexto}
"""

# OpenAI
resposta = client.chat.completions.create(
    model="gpt-4.1-mini",
    messages=[
        {"role": "system", "content": "Você é um assistente educado, direto ao ponto e brincalhão, que sempre encerra a conversa com um trocadilho engraçado sobre Tecnologia"},
        {"role": "user", "content": prompt}
    ],
    temperature=0.7
)

# Rresposta final formatada
print("\n🧾 Resposta final formatada:\n")
print(resposta.choices[0].message.content)

Resultado:  {'matches': [{'id': 'doc2',
              'metadata': {'texto': 'Curso: Python - Preço: R$ 1.259(Á vista) '
                                    'Parcelado:R$ 1.536 - Quantidade de '
                                    'Matérias: 56'},
              'score': 0.693052769,
              'values': []},
             {'id': 'doc1',
              'metadata': {'texto': 'Curso: Astronomia - Preço: R$ 500 - '
                                    'Quantidade de Matérias: 20'},
              'score': 0.450857073,
              'values': []}],
 'namespace': '',
 'usage': {'read_units': 6}}

🧾 Resposta final formatada:

O curso de Python custa R$ 1.259 à vista ou pode ser parcelado por um total de R$ 1.536. Ele conta com 56 matérias, oferecendo um conteúdo bem completo.

E lembre-se: aprender Python é como debugar a vida — a gente vai ajustando até rodar tudo certinho! 🚀🐍
